# Notebook 03 — Exploratory Data Analysis (EDA)

**Objective:** Uncover distributions, trends, and anomalies in the cleaned dataset. Each visualisation closes with a short business takeaway so the notebook reads like an analytical story, not just a chart gallery.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(
    os.environ.get(
        'PROJECT_ROOT',
        Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
    )
).resolve()

def first_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

RAW_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'raw' / 'investments_VC.csv',
    PROJECT_ROOT / 'investments_VC.csv',
    Path('/content/investments_VC.csv'),
)

CLEAN_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'processed' / 'startups_cleaned.csv',
    PROJECT_ROOT / 'startups_cleaned.csv',
    Path('/content/startups_cleaned.csv'),
)

SCREENSHOT_DIR = PROJECT_ROOT / 'tableau' / 'screenshots'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    _colab_files = None
    IN_COLAB = False

def maybe_download(path: Path) -> None:
    if IN_COLAB:
        _colab_files.download(str(path))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

COLORS = {'closed': '#E74C3C', 'operating': '#2ECC71', 'acquired': '#3498DB', 'ipo': '#F39C12'}

df = pd.read_csv(CLEAN_DATA_PATH, low_memory=False)
df = df.rename(columns={c: c.lower() if c.startswith('round_') else c for c in df.columns})

print(f'Loaded cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('Status distribution:')
print(df['status'].value_counts().to_string())

def save_chart(fig, filename: str) -> Path:
    output_path = SCREENSHOT_DIR / filename
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    maybe_download(output_path)
    return output_path


In [ ]:
status_counts = df['status'].value_counts()
status_pct = (status_counts / status_counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(
    status_counts.index,
    status_counts.values,
    color=[COLORS.get(s, '#95A5A6') for s in status_counts.index],
    edgecolor='white',
    linewidth=1.5,
)
for bar, val in zip(bars, status_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 120, f'{val:,}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Startup Count by Status', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Number of Startups')
axes[0].set_xlabel('Status')
axes[1].pie(
    status_pct.values,
    labels=[f'{s}\n{p}%' for s, p in status_pct.items()],
    colors=[COLORS.get(s, '#95A5A6') for s in status_pct.index],
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
axes[1].set_title('Startup Outcome Distribution (%)', fontweight='bold', fontsize=13)
plt.suptitle('Figure 1: What Happens to Startups After Receiving VC Funding?', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save_chart(fig, 'eda_01_status_distribution.png')
plt.show()

## Insight
Most startups in the cleaned sample are still marked `operating`, while only about **5.1%** are labelled `closed`. That makes failure the minority class, which is important later when we interpret statistical tests and any predictive model.

In [ ]:
df_rounds = df[df['funding_rounds'] <= 10].copy()
closure_by_rounds = df_rounds.groupby('funding_rounds')['is_closed'].mean().reset_index()
closure_by_rounds.columns = ['funding_rounds', 'failure_rate']
closure_by_rounds['failure_rate_pct'] = (closure_by_rounds['failure_rate'] * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    closure_by_rounds['funding_rounds'].astype(str),
    closure_by_rounds['failure_rate_pct'],
    color='#E74C3C',
    alpha=0.85,
    edgecolor='white',
    linewidth=1.5,
)
for bar, val in zip(bars, closure_by_rounds['failure_rate_pct']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2, f'{val}%', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Figure 2: Startup Failure Rate by Number of Funding Rounds', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Funding Rounds Received')
ax.set_ylabel('Failure Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
plt.tight_layout()
save_chart(fig, 'eda_02_failure_by_rounds.png')
plt.show()

## Insight
Failure is highest among startups with just **one funding round (about 6.1%)** and generally falls as companies raise more rounds. Repeat funding looks less like a vanity metric and more like an external signal that investors still believe the company is progressing.

In [ ]:
sector_stats = df.groupby('market').agg(total=('is_closed', 'count'), closed=('is_closed', 'sum')).reset_index()
sector_stats = sector_stats[sector_stats['total'] >= 50]
sector_stats['failure_rate'] = (sector_stats['closed'] / sector_stats['total'] * 100).round(1)
sector_stats = sector_stats.sort_values('failure_rate', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(13, 8))
bars = ax.barh(
    sector_stats['market'],
    sector_stats['failure_rate'],
    color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(sector_stats))),
    edgecolor='white',
    linewidth=1,
)
for bar, val in zip(bars, sector_stats['failure_rate']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2, f'{val}%', va='center', fontsize=9)
ax.set_title('Figure 3: Top 20 Markets by Startup Failure Rate (min. 50 startups)', fontweight='bold', fontsize=13)
ax.set_xlabel('Failure Rate (%)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
plt.tight_layout()
save_chart(fig, 'eda_03_failure_by_market.png')
plt.show()

## Insight
Sector risk is not evenly distributed. Consumer-facing niches such as `Curated Web` and `Games` sit near the top of the failure-rate ranking, which means portfolio risk depends partly on *where* a startup plays, not only on how much it raises.

In [ ]:
df_fund = df[df['funding_total_usd'] > 0].copy()
df_fund['log_funding'] = np.log10(df_fund['funding_total_usd'])

fig, ax = plt.subplots(figsize=(13, 5))
for status in ['closed', 'operating', 'acquired']:
    subset = df_fund[df_fund['status'] == status]['log_funding']
    if len(subset) > 1:
        subset.plot.kde(
            ax=ax,
            label=f'{status.title()} (n={len(subset):,})',
            color=COLORS.get(status),
            linewidth=2.5,
        )
ax.set_title('Figure 4: Total Funding Distribution by Startup Status (Log₁₀ Scale)', fontweight='bold', fontsize=13)
ax.set_xlabel('log₁₀(Total Funding USD)')
ax.set_ylabel('Density')
ax.legend(title='Status')
ax.set_xticks([3, 4, 5, 6, 7, 8, 9])
ax.set_xticklabels(['$1K', '$10K', '$100K', '$1M', '$10M', '$100M', '$1B'])
plt.tight_layout()
save_chart(fig, 'eda_04_funding_distribution.png')
plt.show()

## Insight
The closed-startup funding curve is visibly left-shifted relative to operating and acquired firms. That pattern suggests that undercapitalisation is not just anecdotal; it shows up in the full funding distribution as a recurring structural gap.

In [ ]:
yearly = df[df['founded_year'].between(1990, 2014)].groupby('founded_year').agg(total=('is_closed', 'count'), closed=('is_closed', 'sum')).reset_index()
yearly['failure_rate'] = (yearly['closed'] / yearly['total'] * 100).round(2)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(yearly['founded_year'], yearly['total'], alpha=0.3, color='#3498DB', label='Total Startups')
ax2.plot(yearly['founded_year'], yearly['failure_rate'], color='#E74C3C', linewidth=2.5, marker='o', markersize=4, label='Failure Rate %')
ax1.set_xlabel('Founding Year')
ax1.set_ylabel('Number of Startups', color='#3498DB')
ax2.set_ylabel('Failure Rate (%)', color='#E74C3C')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax1.set_title('Figure 5: Startup Volumes and Failure Rate by Founding Year (1990–2014)', fontweight='bold', fontsize=13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
save_chart(fig, 'eda_05_failure_trend.png')
plt.show()

## Insight
Failure rates rise sharply for startups founded in the mid-to-late 2000s, with **2007** standing out as the peak year in this cleaned sample. Time matters here: startup risk is shaped not only by company choices, but also by the market conditions surrounding each cohort.

In [ ]:
country_stats = df.groupby('country_code').agg(total=('is_closed', 'count'), closed=('is_closed', 'sum')).reset_index()
country_stats = country_stats[country_stats['total'] >= 100]
country_stats['failure_rate'] = (country_stats['closed'] / country_stats['total'] * 100).round(1)
country_stats = country_stats.sort_values('failure_rate', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(
    country_stats['country_code'],
    country_stats['failure_rate'],
    color=plt.cm.Reds(np.linspace(0.4, 0.9, len(country_stats))),
    edgecolor='white',
    linewidth=1.5,
)
for bar, val in zip(bars, country_stats['failure_rate']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2, f'{val}%', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Figure 6: Startup Failure Rate by Country (min. 100 startups)', fontweight='bold', fontsize=13)
ax.set_xlabel('Country Code')
ax.set_ylabel('Failure Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
plt.tight_layout()
save_chart(fig, 'eda_06_failure_by_country.png')
plt.show()

## Insight
Geography also changes the baseline risk. Countries such as `RUS`, `ISR`, and `GBR` appear above the global average in this filtered sample, reminding us that startup outcomes depend partly on ecosystem maturity and not only on company-level execution.

In [ ]:
corr_cols = ['is_closed', 'funding_rounds', 'funding_total_usd', 'days_to_first_funding', 'funding_duration_days', 'avg_funding_per_round']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    df[corr_cols].corr(),
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    ax=ax,
)
ax.set_title('Correlation Matrix — Key Numerical Features', fontsize=14, pad=12)
plt.tight_layout()
save_chart(fig, 'eda_07_correlation_heatmap.png')
plt.show()

## Insight
The heatmap shows that closure is only weakly correlated with any single numeric field, which is exactly why a multivariate view matters. Funding rounds, funding duration, and capital efficiency all move in the expected protective direction, but none alone fully explains startup failure.

In [ ]:
series_a_failure = df.groupby('reached_series_a')['is_closed'].mean().reset_index()
series_a_failure['label'] = series_a_failure['reached_series_a'].map({0: 'No Series A', 1: 'Reached Series A'})

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    series_a_failure['label'],
    series_a_failure['is_closed'] * 100,
    color=['#e74c3c', '#2ecc71'],
    edgecolor='white',
    width=0.5,
)
ax.set_ylabel('Failure Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title('Series A as a Survival Inflection Point', fontsize=14)
plt.tight_layout()
save_chart(fig, 'eda_08_series_a_survival.png')
plt.show()

## Insight
Startups that reached Series A show a slightly lower failure rate than those that did not (**4.7% vs 5.2%** in this current ETL output). The gap is directionally helpful, but in this sample it reads as a supporting signal rather than a standalone guarantee of survival.